# ETL Consolidado: RHR, Movimiento y Hormonas

Crea un dataframe con una observación por `day_in_study` por sujeto para el intervalo 2022.

Variables incluidas:
- Movimiento (movement_per_day_percentage)
- Frecuencia cardíaca en reposo (métricas dinámicas)
- Hormonas (LH, Estrógeno)

## 1. Setup y Carga de Datos

In [11]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Definir rutas
data_path = Path("/Users/daragama/DBS/mcphases1.0.0")
output_path = Path("/Users/daragama/Documents/ProyectosVarios/DTW_mcphases/data")

# Crear carpeta de output si no existe
output_path.mkdir(parents=True, exist_ok=True)

print(f"Data path: {data_path}")
print(f"Output path: {output_path}")

Data path: /Users/daragama/DBS/mcphases1.0.0
Output path: /Users/daragama/Documents/ProyectosVarios/DTW_mcphases/data


In [121]:
# Cargar CSVs
df_exercise = pd.read_csv(data_path / "active_minutes.csv")
df_rhr = pd.read_csv(data_path / "resting_heart_rate.csv")
df_heart_rate = pd.read_csv(data_path / "heart_rate.csv")  # Para métricas dinámicas
df_hormones = pd.read_csv(data_path / "hormones_and_selfreport.csv")
df_wrist_temperature = pd.read_csv(data_path / "wrist_temperature.csv")
df_computed_temperature = pd.read_csv(data_path / "computed_temperature.csv")

print("Datos cargados:")
print(f"  - Exercise: {df_exercise.shape}")
print(f"  - RHR (diario): {df_rhr.shape}")
print(f"  - Heart Rate (intradiario): {df_heart_rate.shape}")
print(f"  - Hormones: {df_hormones.shape}")

Datos cargados:
  - Exercise: (5552, 8)
  - RHR (diario): (13737, 6)
  - Heart Rate (intradiario): (63100276, 7)
  - Hormones: (5659, 22)


## 2. Procesamiento de Movimiento (Active Minutes)

In [13]:
# Revisar estructura
df_exercise.head()

,id,study_interval,is_weekend,day_in_study,sedentary,lightly,moderately,very
0,1,2022,True,1,753.0,64,0,0
1,1,2022,False,2,855.0,74,0,0
2,1,2022,False,3,751.0,134,18,7
3,1,2022,False,4,905.0,86,0,0
4,1,2022,False,5,1430.0,10,0,0


In [14]:
# Filtrar intervalo 2022 y preparar movimiento
df_movement = (
    df_exercise[df_exercise["study_interval"] == 2022]
    .copy()
)

# Calcular movement_per_day_percentage si no existe
if "movement_per_day_percentage" not in df_movement.columns:
    df_movement["sedentary_percentage"] = (
        df_movement["sedentary"] /
        df_movement[["sedentary", "lightly", "moderately", "very"]].sum(axis=1)
    )
    df_movement["movement_per_day_percentage"] = 1 - df_movement["sedentary_percentage"]

# Asegurar una observación por day_in_study por id
df_movement_daily = (
    df_movement
    .groupby(["id", "day_in_study"], as_index=False)
    .agg({"movement_per_day_percentage": "mean"})
    .rename(columns={"movement_per_day_percentage": "movement_pct"})
)

print(f"Movimiento diario: {df_movement_daily.shape}")
print(f"Sujetos únicos: {df_movement_daily['id'].nunique()}")
df_movement_daily.head()

Movimiento diario: (3698, 3)
Sujetos únicos: 42


,id,day_in_study,movement_pct
0,1,1,0.078335
1,1,2,0.079656
2,1,3,0.174725
3,1,4,0.086781
4,1,5,0.006944


## 3. Procesamiento de Frecuencia Cardíaca en Reposo (RHR)

In [99]:
# Revisar estructura
df_rhr.head()

,id,study_interval,is_weekend,day_in_study,value,error
0,1,2022,True,1,74.785346,100.000000
1,1,2022,False,2,80.407307,29.833838
2,1,2022,False,3,84.686869,24.267298
3,1,2022,False,4,83.852219,10.344703
4,1,2022,False,5,0.000000,0.000000


In [100]:
# Revisar estructura y nulos
print("RHR - Estructura y nulos:")
print(f"Shape: {df_rhr.shape}")
print(f"\nNulos por columna:")
print(df_rhr.isnull().sum())
print(f"\nPrimeras filas:")
print(df_rhr.head())

RHR - Estructura y nulos:
Shape: (13737, 6)

Nulos por columna:
id                0
study_interval    0
is_weekend        0
day_in_study      0
value             0
error             0
dtype: int64

Primeras filas:
   id  study_interval  is_weekend  day_in_study      value       error
0   1            2022        True             1  74.785346  100.000000
1   1            2022       False             2  80.407307   29.833838
2   1            2022       False             3  84.686869   24.267298
3   1            2022       False             4  83.852219   10.344703
4   1            2022       False             5   0.000000    0.000000


In [101]:
# Filtrar intervalo 2022 y preparar RHR diario
df_rhr_2022 = df_rhr[df_rhr["study_interval"] == 2022].copy()

# RHR ya tiene una medición por día, pero verificar duplicados
print(f"RHR 2022: {df_rhr_2022.shape}")
print(f"Sujetos únicos: {df_rhr_2022['id'].nunique()}")
print(f"\nDuplicados (id, day_in_study): {df_rhr_2022.duplicated(subset=['id', 'day_in_study']).sum()}")

# Asegurar una observación por día por id (tomar la primera si hay duplicados)
df_rhr_daily = (
    df_rhr_2022
    .groupby(['id', 'day_in_study'], as_index=False)
    .first()
    .rename(columns={'value': 'rhr_value'}, inplace=False)
)

print(f"\nRHR diario final: {df_rhr_daily.shape}")
print(f"Nulos en value: {df_rhr_daily['rhr_value'].isnull().sum()}")
df_rhr_daily.head()

RHR 2022: (3698, 6)
Sujetos únicos: 42

Duplicados (id, day_in_study): 0

RHR diario final: (3698, 6)
Nulos en value: 0


,id,day_in_study,study_interval,is_weekend,rhr_value,error
0,1,1,2022,True,74.785346,100.000000
1,1,2,2022,False,80.407307,29.833838
2,1,3,2022,False,84.686869,24.267298
3,1,4,2022,False,83.852219,10.344703
4,1,5,2022,False,0.000000,0.000000


## 3.5 Métricas Dinámicas de Heart Rate (intradiario)

In [102]:
# Revisar estructura y nulos
print("Heart Rate - Estructura y nulos:")
print(f"Shape: {df_heart_rate.shape}")
print(f"\nNulos por columna:")
print(df_heart_rate.isnull().sum())
print(f"\nPrimeras filas:")
print(df_heart_rate.head())

Heart Rate - Estructura y nulos:
Shape: (63100276, 7)

Nulos por columna:
id                0
study_interval    0
is_weekend        0
day_in_study      0
timestamp         0
bpm               0
confidence        0
dtype: int64

Primeras filas:
   id  study_interval  is_weekend  day_in_study timestamp  bpm  confidence
0   1            2022        True             1  01:22:09   75           1
1   1            2022        True             1  01:22:14   79           1
2   1            2022        True             1  01:22:19   80           1
3   1            2022        True             1  01:22:24   81           1
4   1            2022        True             1  01:22:29   80           1


In [103]:
# Filtrar intervalo 2022
df_hr_2022 = df_heart_rate[df_heart_rate["study_interval"] == 2022].copy()

print(f"Heart Rate 2022: {df_hr_2022.shape}")
print(f"Sujetos únicos: {df_hr_2022['id'].nunique()}")
print(f"Nulos en bpm: {df_hr_2022['bpm'].isnull().sum()}")

Heart Rate 2022: (39812695, 7)
Sujetos únicos: 41
Nulos en bpm: 0


| Métrica                | Qué captura                          |
| ---------------------- | ------------------------------------ |
| `hr_mean`              | Nivel basal de BPM                   |
| `hr_sd`                | Variabilidad global                  |
| `hr_mean_abs_delta`    | Intensidad promedio de fluctuaciones |
| `hr_sd_delta`          | Irregularidad de las fluctuaciones   |
| `hr_path_length`       | Movimiento total acumulado           |
| `hr_direction_changes` | Frecuencia de oscilaciones           |


In [104]:
import numpy as np
import pandas as pd

import numpy as np
import pandas as pd

def calculate_hr_dynamics(group):

    bpm = group["bpm"].dropna().to_numpy()

    if len(bpm) < 2:
        return pd.Series({
            "hr_mean": np.nan,
            "hr_sd": np.nan,
            "hr_mean_abs_delta": np.nan,
            "hr_sd_delta": np.nan,
            "hr_direction_changes": np.nan,
            "hr_range": np.nan,
            "hr_cv": np.nan,
            "hr_path_length": np.nan
        })

    delta = np.diff(bpm)
    abs_delta = np.abs(delta)

    return pd.Series({
        "hr_mean": np.mean(bpm),
        "hr_sd": np.std(bpm, ddof=1),
        "hr_mean_abs_delta": np.mean(abs_delta),
        "hr_sd_delta": np.std(delta, ddof=1),
        "hr_direction_changes": np.sum(
            np.diff(np.sign(delta)) != 0
        ),
        "hr_range": np.max(bpm) - np.min(bpm),
        "hr_cv": np.std(bpm, ddof=1) / np.mean(bpm),
        "hr_path_length": np.sum(abs_delta)
    })

# Agrupar por id y day_in_study
df_hr_daily_metrics = (
    df_hr_2022
    .groupby(['id', 'day_in_study'], as_index=False)
    .apply(calculate_hr_dynamics, include_groups=False)
)

print(f"Heart Rate dinámico diario: {df_hr_daily_metrics.shape}")
print(f"Sujetos únicos: {df_hr_daily_metrics['id'].nunique()}")
print(f"\nNulos por métrica:")
print(df_hr_daily_metrics.isnull().sum())
df_hr_daily_metrics.head()

Heart Rate dinámico diario: (3483, 10)
Sujetos únicos: 41

Nulos por métrica:
id                      0
day_in_study            0
hr_mean                 0
hr_sd                   0
hr_mean_abs_delta       0
hr_sd_delta             0
hr_direction_changes    0
hr_range                0
hr_cv                   0
hr_path_length          0
dtype: int64


,id,day_in_study,hr_mean,hr_sd,hr_mean_abs_delta,hr_sd_delta,hr_direction_changes,hr_range,hr_cv,hr_path_length
0,1,1,77.641853,11.317769,1.556217,2.189369,4091.0,69.0,0.145769,12291.0
1,1,2,95.732149,20.507059,2.033951,3.192844,4487.0,111.0,0.214213,19111.0
2,1,3,104.158134,19.085520,2.169022,3.400073,5028.0,118.0,0.183236,21944.0
3,1,4,87.039370,11.372742,1.557387,2.094298,3542.0,66.0,0.130662,10679.0
4,1,5,102.900407,24.700893,2.384493,4.018943,3203.0,109.0,0.240047,16422.0


### PCA on HR

In [105]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X = df_hr_daily_metrics[["hr_mean", "hr_sd", "hr_mean_abs_delta", "hr_sd_delta", "hr_direction_changes"]].copy()
X = X.interpolate(limit_direction="both")
X_scaled = StandardScaler().fit_transform(X)
pca = PCA()
X_pca = pca.fit_transform(X_scaled)
explained = pd.DataFrame({
    "PC": range(1, len(pca.explained_variance_ratio_) + 1),
    "Explained_Var": pca.explained_variance_ratio_,
    "Cumulative": pca.explained_variance_ratio_.cumsum()
})

print(explained)

   PC  Explained_Var  Cumulative
0   1       0.478528    0.478528
1   2       0.236977    0.715505
2   3       0.166952    0.882457
3   4       0.105796    0.988253
4   5       0.011747    1.000000


In [106]:
pca = PCA(n_components=3)

pcs = pca.fit_transform(X_scaled)

df_hr_daily_metrics["hr_pc1"] = pcs[:, 0]
df_hr_daily_metrics["hr_pc2"] = pcs[:, 1]
df_hr_daily_metrics["hr_pc3"] = pcs[:, 2]

df_hr_daily_pca = df_hr_daily_metrics[["id", "day_in_study", "hr_pc1", "hr_pc2", "hr_pc3"]].copy()

In [107]:
df_hr_daily_pca.head()

,id,day_in_study,hr_pc1,hr_pc2,hr_pc3
0,1,1,-0.087502,-1.584184,0.756453
1,1,2,4.716075,0.174529,0.696757
2,1,3,5.546794,0.622128,1.111908
3,1,4,0.130233,-1.319226,1.926947
4,1,5,8.362226,-0.195472,0.846024


## 4. Procesamiento de Hormonas

In [81]:
# Revisar estructura y nulos
print("Hormonas - Estructura y nulos:")
print(f"Shape: {df_hormones.shape}")
print(f"\nNulos por columna:")
print(df_hormones.isnull().sum())
print(f"\nColumnas: {df_hormones.columns.tolist()}")

Hormonas - Estructura y nulos:
Shape: (5659, 22)

Nulos por columna:
id                   0
study_interval       0
is_weekend           0
day_in_study         0
phase                1
lh                 320
estrogen           321
pdg               3795
flow_volume       2470
flow_color        2465
appetite          2329
exerciselevel     2329
headaches         2331
cramps            2332
sorebreasts       2332
fatigue           2328
sleepissue        2330
moodswing         2339
stress            2332
foodcravings      2332
indigestion       2334
bloating          2331
dtype: int64

Columnas: ['id', 'study_interval', 'is_weekend', 'day_in_study', 'phase', 'lh', 'estrogen', 'pdg', 'flow_volume', 'flow_color', 'appetite', 'exerciselevel', 'headaches', 'cramps', 'sorebreasts', 'fatigue', 'sleepissue', 'moodswing', 'stress', 'foodcravings', 'indigestion', 'bloating']


In [111]:
# Filtrar intervalo 2022 y seleccionar LH y Estrógeno
df_hormones_2022 = df_hormones[df_hormones["study_interval"] == 2022].copy()

print(f"Hormonas 2022: {df_hormones_2022.shape}")
print(f"Sujetos únicos: {df_hormones_2022['id'].nunique()}")

# Seleccionar columnas relevantes (LH y Estrógeno)
hormone_cols = [col for col in ["lh", "estrogen", "phase"]]

print(f"\nHormonas disponibles: {hormone_cols}")
print(f"Nulos por hormona:")
for col in hormone_cols:
    print(f"  {col}: {df_hormones_2022[col].isnull().sum()}")

Hormonas 2022: (3698, 22)
Sujetos únicos: 42

Hormonas disponibles: ['lh', 'estrogen', 'phase']
Nulos por hormona:
  lh: 223
  estrogen: 224
  phase: 1


In [112]:
# Preparar dataframe de hormonas: una medición por day_in_study
# Agrupar y tomar la primera si hay múltiples
df_hormones_daily = (
    df_hormones_2022
    .groupby(["id", "day_in_study"], as_index=False)
    .first()
)

# Seleccionar solo id, day_in_study y variables hormonales
hormones_to_keep = ["id", "day_in_study"] + hormone_cols
df_hormones_daily = df_hormones_daily[hormones_to_keep]

print(f"Hormonas diario: {df_hormones_daily.shape}")
print(f"Sujetos únicos: {df_hormones_daily['id'].nunique()}")
print(f"\nNulos por columna:")
print(df_hormones_daily.isnull().sum())
df_hormones_daily.head()

Hormonas diario: (3698, 5)
Sujetos únicos: 42

Nulos por columna:
id                0
day_in_study      0
lh              223
estrogen        224
phase             1
dtype: int64


,id,day_in_study,lh,estrogen,phase
0,1,1,2.9,94.2,Follicular
1,1,2,1.2,226.3,Follicular
2,1,3,3.5,276.8,Follicular
3,1,4,1.8,322.1,Fertility
4,1,5,4.6,244.9,Fertility


## 5. Temperature

In [122]:
# Revisar estructura y nulos
print("Wrist Temperature - Estructura y nulos:")
print(f"Shape: {df_wrist_temperature.shape}")
print(f"\nNulos por columna:")
print(df_wrist_temperature.isnull().sum())
print(f"\nColumnas: {df_wrist_temperature.columns.tolist()}")

Wrist Temperature - Estructura y nulos:
Shape: (6856019, 6)

Nulos por columna:
id                                0
study_interval                    0
is_weekend                        0
day_in_study                      0
timestamp                         0
temperature_diff_from_baseline    0
dtype: int64

Columnas: ['id', 'study_interval', 'is_weekend', 'day_in_study', 'timestamp', 'temperature_diff_from_baseline']


Nivel:
- temp_mean
- temp_sd
- temp_min
- temp_max
- temp_range

Dinámica:
- temp_mean_abs_delta
- temp_sd_delta
- temp_path_length

Distribución:
- temp_p10
- temp_p90

In [123]:
import numpy as np
import pandas as pd

def calculate_temp_dynamics(group):

    temp = (
        group["temperature_diff_from_baseline"]
        .dropna()
        .to_numpy()
    )

    if len(temp) < 2:
        return pd.Series({
            "temp_mean": np.nan,
            "temp_sd": np.nan,
            "temp_min": np.nan,
            "temp_max": np.nan,
            "temp_range": np.nan,
            "temp_mean_abs_delta": np.nan,
            "temp_sd_delta": np.nan,
            "temp_path_length": np.nan,
            "temp_p10": np.nan,
            "temp_p90": np.nan
        })

    delta = np.diff(temp)

    return pd.Series({
        "temp_mean": np.mean(temp),
        "temp_sd": np.std(temp, ddof=1),
        "temp_min": np.min(temp),
        "temp_max": np.max(temp),
        "temp_range": np.max(temp) - np.min(temp),
        "temp_mean_abs_delta": np.mean(np.abs(delta)),
        "temp_sd_delta": np.std(delta, ddof=1),
        "temp_path_length": np.sum(np.abs(delta)),
        "temp_p10": np.percentile(temp, 10),
        "temp_p90": np.percentile(temp, 90)
    })

In [125]:
df_temp_daily = (
    df_wrist_temperature
    .groupby(["id", "day_in_study"], as_index=False)
    .apply(
        calculate_temp_dynamics,
        include_groups=False
    )
)

In [126]:
temp_features = [
    "temp_mean",
    "temp_sd",
    "temp_min",
    "temp_max",
    "temp_range",
    "temp_mean_abs_delta",
    "temp_sd_delta",
    "temp_path_length",
    "temp_p10",
    "temp_p90"
]

### PCA

In [129]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X = df_temp_daily[temp_features].copy()
X = X.interpolate(limit_direction="both")
X_scaled = StandardScaler().fit_transform(X)


pca = PCA()
X_pca = pca.fit_transform(X_scaled)

explained = pd.DataFrame({
    "PC": range(1, len(pca.explained_variance_ratio_) + 1),
    "Explained_Var": pca.explained_variance_ratio_,
    "Cumulative": pca.explained_variance_ratio_.cumsum()
})

print(explained)

   PC  Explained_Var  Cumulative
0   1   4.112078e-01    0.411208
1   2   2.399360e-01    0.651144
2   3   1.403947e-01    0.791539
3   4   9.990261e-02    0.891441
4   5   5.145594e-02    0.942897
5   6   3.469399e-02    0.977591
6   7   1.305233e-02    0.990643
7   8   7.365442e-03    0.998009
8   9   1.991170e-03    1.000000
9  10   1.305054e-17    1.000000


In [131]:
pca = PCA(n_components=4)

pcs = pca.fit_transform(X_scaled)

df_temp_daily["wtmp_pc1"] = pcs[:, 0]
df_temp_daily["wtmp_pc2"] = pcs[:, 1]
df_temp_daily["wtmp_pc3"] = pcs[:, 2]
df_temp_daily["wtmp_pc4"] = pcs[:, 3]

## 5.2 Computed Temperature

In [132]:
# Revisar estructura y nulos
print("Computed Temperature - Estructura y nulos:")
print(f"Shape: {df_computed_temperature.shape}")
print(f"\nNulos por columna:")
print(df_computed_temperature.isnull().sum())
print(f"\nColumnas: {df_computed_temperature.columns.tolist()}")

Computed Temperature - Estructura y nulos:
Shape: (5575, 14)

Nulos por columna:
id                                                0
study_interval                                    0
is_weekend                                        0
sleep_start_day_in_study                          0
sleep_start_timestamp                             0
sleep_end_day_in_study                            0
sleep_end_timestamp                               0
type                                              0
temperature_samples                               0
nightly_temperature                               0
baseline_relative_sample_sum                    379
baseline_relative_sample_sum_of_squares         379
baseline_relative_nightly_standard_deviation    379
baseline_relative_sample_standard_deviation     379
dtype: int64

Columnas: ['id', 'study_interval', 'is_weekend', 'sleep_start_day_in_study', 'sleep_start_timestamp', 'sleep_end_day_in_study', 'sleep_end_timestamp', 'type', 'temperature_sampl

TODO: temperature samples

## 6. Consolidación del Dataframe Final

In [ ]:
# Merge de movimiento + RHR diario (valor simple)
df_consolidated = (
    df_movement_daily
    .merge(df_rhr_daily[["id", "day_in_study", "rhr_value"]], 
           on=["id", "day_in_study"], how="outer")
)

print(f"After RHR merge: {df_consolidated.shape}")

# Merge con PCA de dinámicas de Heart Rate
df_consolidated = (
    df_consolidated
    .merge(df_hr_daily_pca, 
           on=["id", "day_in_study"], how="outer")
)

print(f"After HR dynamics merge: {df_consolidated.shape}")

# Merge con hormonas
df_consolidated = (
    df_consolidated
    .merge(df_hormones_daily, 
           on=["id", "day_in_study"], how="outer")
)

print(f"After hormones merge: {df_consolidated.shape}")

# merge con wrist temperature
df_consolidated = (
    df_consolidated
    .merge(df_temp_daily[["id", "day_in_study", "wtmp_pc1", "wtmp_pc2", "wtmp_pc3", "wtmp_pc4"]], 
           on=["id", "day_in_study"], how="outer")
)

print(f"After wrist temperature merge: {df_consolidated.shape}")

# Ordenar por id y day_in_study
df_consolidated = df_consolidated.sort_values(["id", "day_in_study"]).reset_index(drop=True)

print(f"\nDataframe consolidado final: {df_consolidated.shape}")
print(f"Sujetos únicos: {df_consolidated['id'].nunique()}")

After RHR merge: (3698, 4)
After HR dynamics merge: (3698, 7)
After hormones merge: (3698, 10)

Dataframe consolidado final: (3698, 10)
Sujetos únicos: 42


In [114]:
# Ver estructura
print("Primeras filas del dataframe consolidado:")
print(df_consolidated.head(10))
print(f"\nColumnas: {df_consolidated.columns.tolist()}")

Primeras filas del dataframe consolidado:
   id  day_in_study  movement_pct  rhr_value    hr_pc1    hr_pc2    hr_pc3  \
0   1             1      0.078335  74.785346 -0.087502 -1.584184  0.756453   
1   1             2      0.079656  80.407307  4.716075  0.174529  0.696757   
2   1             3      0.174725  84.686869  5.546794  0.622128  1.111908   
3   1             4      0.086781  83.852219  0.130233 -1.319226  1.926947   
4   1             5      0.006944   0.000000  8.362226 -0.195472  0.846024   
5   1             6      0.026316  82.077053  4.788206 -0.530856  1.000721   
6   1             7      0.126898  79.335861  0.442856 -0.682325  1.167039   
7   1             8      0.102778   0.000000  2.520140 -1.910666  3.307995   
8   1             9      0.071429  76.736727  6.235750 -0.571468 -0.012039   
9   1            10      0.074481  76.589839  4.269572  0.353902  0.289978   

    lh  estrogen       phase  
0  2.9      94.2  Follicular  
1  1.2     226.3  Follicular  
2  3.5

In [115]:
# Resumen detallado
print("="*60)
print("RESUMEN DE CALIDAD DE DATOS")
print("="*60)

print("\nTamaño final:")
print(f"  - Filas (observaciones): {df_consolidated.shape[0]}")
print(f"  - Columnas (variables): {df_consolidated.shape[1]}")
print(f"  - Sujetos únicos: {df_consolidated['id'].nunique()}")

print("\nMissing values por columna:")
missing_summary = pd.DataFrame({
    'Column': df_consolidated.columns,
    'Missing_Count': df_consolidated.isnull().sum(),
    'Missing_Pct': (df_consolidated.isnull().sum() / len(df_consolidated) * 100).round(2)
})
print(missing_summary.to_string(index=False))

print("\nEstructura del dataframe:")
print(df_consolidated.info())

print("\nEstadísticas descriptivas:")
print(df_consolidated.describe())

RESUMEN DE CALIDAD DE DATOS

Tamaño final:
  - Filas (observaciones): 3698
  - Columnas (variables): 10
  - Sujetos únicos: 42

Missing values por columna:
      Column  Missing_Count  Missing_Pct
          id              0         0.00
day_in_study              0         0.00
movement_pct              0         0.00
   rhr_value              0         0.00
      hr_pc1            215         5.81
      hr_pc2            215         5.81
      hr_pc3            215         5.81
          lh            223         6.03
    estrogen            224         6.06
       phase              1         0.03

Estructura del dataframe:
<class 'pandas.DataFrame'>
RangeIndex: 3698 entries, 0 to 3697
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            3698 non-null   int64  
 1   day_in_study  3698 non-null   int64  
 2   movement_pct  3698 non-null   float64
 3   rhr_value     3698 non-null   float64
 4   hr_pc1 

In [116]:
# Estadísticas descriptivas
print("Estadísticas descriptivas:")
df_consolidated.describe()

Estadísticas descriptivas:


,id,day_in_study,movement_pct,rhr_value,hr_pc1,hr_pc2,hr_pc3,lh,estrogen
count,3698.000000,3698.000000,3698.000000,3698.000000,3.483000e+03,3.483000e+03,3.483000e+03,3475.000000,3474.000000
mean,25.287182,44.989454,0.200525,53.689706,-1.632025e-17,-8.160123e-18,1.632025e-17,5.335799,138.674036
std,15.076875,25.954792,0.133871,28.639029,1.547038e+00,1.088680e+00,9.137834e-01,6.884436,111.993602
min,1.000000,1.000000,0.000000,0.000000,-5.514762e+00,-6.395343e+00,-2.339618e+00,0.000000,0.000000
25%,12.000000,23.000000,0.106320,55.354611,-8.829650e-01,-5.408101e-01,-6.136257e-01,2.500000,67.325000
50%,24.000000,45.000000,0.194460,64.830880,-2.476772e-01,2.086763e-02,-9.325828e-02,3.600000,108.450000
75%,40.000000,67.000000,0.281515,71.789644,5.703461e-01,5.724979e-01,5.590047e-01,5.500000,174.475000
max,50.000000,90.000000,1.000000,88.088963,1.542745e+01,5.271513e+00,6.069407e+00,97.000000,640.000000


In [117]:
df_consolidated.head()

,id,day_in_study,movement_pct,rhr_value,hr_pc1,hr_pc2,hr_pc3,lh,estrogen,phase
0,1,1,0.078335,74.785346,-0.087502,-1.584184,0.756453,2.9,94.2,Follicular
1,1,2,0.079656,80.407307,4.716075,0.174529,0.696757,1.2,226.3,Follicular
2,1,3,0.174725,84.686869,5.546794,0.622128,1.111908,3.5,276.8,Follicular
3,1,4,0.086781,83.852219,0.130233,-1.319226,1.926947,1.8,322.1,Fertility
4,1,5,0.006944,0.000000,8.362226,-0.195472,0.846024,4.6,244.9,Fertility


## 7. Guardar Resultado

In [118]:
# Verificar una observación por día por id antes de guardar
duplicates = df_consolidated.duplicated(subset=['id', 'day_in_study'], keep=False)
if duplicates.sum() > 0:
    print(f"ADVERTENCIA: Hay {duplicates.sum()} registros duplicados (id, day_in_study)")
    print("Primeros duplicados:")
    print(df_consolidated[duplicates].head())
else:
    print("✓ Confirmado: Una observación por día por sujeto")

# Guardar CSV
output_file = output_path / "mcphases_consolidated_2022.csv"
df_consolidated.to_csv(output_file, index=False)

print(f"\n✓ Archivo guardado en: {output_file}")
print(f"✓ Shape final: {df_consolidated.shape}")

✓ Confirmado: Una observación por día por sujeto

✓ Archivo guardado en: /Users/daragama/Documents/ProyectosVarios/DTW_mcphases/data/mcphases_consolidated_2022.csv
✓ Shape final: (3698, 10)


In [119]:
# visualizar 12 series de un sujeto de ejemplo
